# 知能情報基礎演習
# オープン語彙物体認識による身近な物体の検出

**クローズド物体検出器（YOLO26）** と **オープン語彙物体認識器（YOLOE）** を実際に動かし、
身近な物体をどこまで検出できるかを調べる。

**Google Colab で動かす。** GPU は無くてよい（CPU のままで十分終わる）。
自分のパソコンで動かしたい人は、いちばん下の「付録B」を見ること。

## このノートブックの構成

| 節 | 内容 | 区分 |
|----|------|------|
| 0 | 準備（インストール・モデル・共通関数・動かし方） | 準備 |
| 1 | YOLO26 を動かし、何が検出できるか確かめる | **【標準】** |
| 2 | 同じ画像に YOLOE を試し、並べて比較する | **【標準】** |
| 3 | YOLO26 が検出できない物を3つ探し、YOLOE で試す | **【標準】** ← **ここまでで【標準】は完了** |
| 4 | 語彙を振って、得意・不得意を調べる | **【発展】** |
| 5 | MS COCO のクラスとの「意味の近さ」を測る | **【発展】** |
| 6 | リアルタイムで動くものを作る | **【発展】**（任意） |
| 付録A | 結果の保存 | — |
| 付録B | 自分のパソコンで動かす | — |

**【標準】は 1〜3 節、【発展】は 4〜6 節である。**（6節は任意）
区切りのところに、進み具合を確かめるためのセルを置いてある。

## この演習でやること

**セルを上から順に実行すれば進められる。** 標準課題のコードは最初から全部書いてあるので、
プログラミングでつまずいて先に進めない、ということにはならない。

自分たちで決めるのは、**どんなデータを入れるか**である。

- どんな画像を撮るか（2節）
- どの3つの物体を対象にするか（3節）
- どんな語を試すか（4節）

同じプログラムでも、**入れるデータを変えれば結果は変わる。**
その変わり方を観察して考察することが、この演習の中身である。

### ただし、コードはしっかり読むこと

実行するだけで数字と図は出る。しかし**中身を知らずに出た数字は、考察に使えない。**

また， **観察と考察に時間を使うこと。**

## この課題で出てくる言葉

| 言葉 | 意味 |
|------|------|
| クローズド物体認識 | 学習時に決めたクラスしか検出できない、従来の物体検出 |
| オープン語彙物体認識 | 使うときに言葉で「何を検出したいか」を指定できる |
| クラス | 検出器が扱う物体の種類 |
| マスク | 画素単位での領域指定。`-seg` のモデルを使うと出力される |
| スコア（確信度） | その判断がどれだけ確かだとモデルが考えているか（0〜1）。確率ではない |
| 閾値（しきいち） | 「これ以上のスコアなら検出したことにする」と決める境目の値 |
| MS COCO | 物体検出でよく使われるデータセット。80クラスが定義されている |
| 意味の空間 | 画像と単語を同じ場所に対応づけた空間。近さで「意味の近さ」を測る |
| コサイン類似度 | 2つのベクトルの向きがどれだけ揃っているか。1に近いほど似ている |

## 撮影のときに必ず守ること

- **店舗・商業施設の中では撮影しない。** コンビニ・スーパー・書店などは、ほとんどが店内撮影を禁止している。
  棚を撮りたいなら、自宅・教室・許可を得た場所にする。
- **個人情報が写り込む物を対象にしない。** 学生証・名札・保険証・郵便物・PCやスマホの画面など。
- 屋外（自転車置き場・食堂など）では、 **人と、番号や氏名が読み取れるもの** が写らない角度で撮る。
  防犯登録番号やナンバープレートに注意。写ってしまったら撮り直す。
- 画像とノートブックは、**指定された提出先以外には置かないこと** （個人のクラウドや GitHub に上げない）。

## ライセンス

- YOLO26 / YOLOE / Ultralytics は **AGPL-3.0**。授業での利用とレポートへの掲載は問題ない。
  ただし AGPL-3.0 には、ふつうのオープンソースライセンスにない条項がある。
  - **改変して配布する場合**、同じライセンスでソースを公開する義務が生じる。
  - **ネットワーク越しにサービスとして提供した場合にも**、同じ義務が生じる（§13）。Web デモの公開はこれに該当する。
  - このノートブック自体が ultralytics を使っているので、**GitHub や個人ブログに公開する場合も対象**になる。
  - → **公開したくなったら、必ず教員に相談すること。**
- 初回に自動で落ちてくる MobileCLIP のテキストエンコーダは Apple の別ライセンス。
- 論文: Wang et al., *YOLOE: Real-Time Seeing Anything*, arXiv:2503.07465 (2025).
  Jocher et al., *Ultralytics YOLO26*, arXiv:2606.03748 (2026).

---
# 0. 準備

## 0-1. インストール

1. このファイルを Google ドライブに置き、**Colaboratory で開く**。
2. 下のセルを実行する。

> **`ultralytics` は 8.4.0 以上が必要。** それ以前のバージョンでは 26 系のモデルが読めない。
> CLIP は「言葉で指定する」機能に必要で、YOLOE が初回に自動で入れようとするが、
> git が要るうえ失敗しやすいので、ここで先に入れておく。

In [ ]:
!pip -q install -U "ultralytics>=8.4.0" japanize-matplotlib
!pip -q install git+https://github.com/ultralytics/CLIP.git

import sys, platform
import torch, ultralytics
from packaging.version import Version

print("Python      :", sys.version.split()[0], "/", platform.platform())
print("ultralytics :", ultralytics.__version__)
print("torch       :", torch.__version__)

assert Version(ultralytics.__version__) >= Version("8.4.0"), \
    "ultralytics が古い。上のセルを実行し直し、ランタイムを再起動すること。"

try:
    import clip
    print("CLIP        : OK")
except ImportError:
    print("!! CLIP が入っていません。上のセルをもう一度実行すること。")

if torch.cuda.is_available():
    DEVICE = 0
    print("device      : GPU (", torch.cuda.get_device_name(0), ")")
else:
    DEVICE = "cpu"
    print("device      : cpu  ← 動くが遅い。数十枚なら十分に終わる。")

## 0-2. 使うモデル

**名前が1文字違いなので、取り違えないこと。**

| 変数 | ファイル | 種類 |
|------|----------|------|
| `YOLO26_MODEL` | `yolo26s-seg.pt` | **クローズド**。MS COCO の80クラスだけ |
| `YOLOE_MODEL` | `yolo**e**-26s-seg.pt` | **オープン語彙**。言葉で指定できる |

大きさは `n < s < m < l < x` の順。**`s` を使う。**
`n` は小さすぎて、「検出できなかったのは語彙のせいか、モデルが小さいせいか」が分からなくなる。

In [ ]:
YOLO26_MODEL = "yolo26s-seg.pt"       # クローズド（COCO 80クラス）
YOLOE_MODEL  = "yoloe-26s-seg.pt"     # オープン語彙

IMGSZ = 640      # CPU で遅ければ 512 や 416 に下げる
CONF  = 0.25     # このスコア以上を「検出した」とみなす

## 0-3. ライブラリの読み込みと日本語フォント

In [ ]:
from pathlib import Path
import glob, math
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO, YOLOE

font_ok = False
try:
    import japanize_matplotlib
    font_ok = True
except Exception:
    import matplotlib.font_manager as fm
    for cand in ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP"]:
        if any(f.name == cand for f in fm.fontManager.ttflist):
            plt.rcParams["font.family"] = cand; font_ok = True; break
plt.rcParams["axes.unicode_minus"] = False
print("日本語フォント:", "OK" if font_ok else "設定できず（図の日本語が □ になる）")

## 0-4. 共通関数

**中身を読んでおくこと。** 何をしているか分からないまま結果だけ眺めても、考察は書けない。

In [ ]:
_models = {}

def get_yolo26():
    if "closed" not in _models:
        print("loading", YOLO26_MODEL, "...")
        _models["closed"] = YOLO(YOLO26_MODEL)
    return _models["closed"]

def get_yoloe():
    if "open" not in _models:
        print("loading", YOLOE_MODEL, "...")
        _models["open"] = YOLOE(YOLOE_MODEL)
    return _models["open"]

ROOT = Path("images")          # 画像を置くフォルダ
ROOT.mkdir(exist_ok=True)

def list_images(d):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")
    found = sorted(sum([glob.glob(str(Path(d) / e)) for e in exts], []))
    heic = sorted(sum([glob.glob(str(Path(d) / e)) for e in ("*.heic", "*.HEIC")], []))
    if heic:
        print(f"!! {Path(d).name}: HEIC 形式が {len(heic)} 枚あります。読み込めません。")
        print("!! iPhone の 設定 → カメラ → フォーマット → 互換性優先 にしてから撮り直すこと。")
    return found

def to_rgb(bgr):
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def dets_of(result, conf_thr=0.0):
    # 1枚分の検出結果を辞書のリストにする
    b = result.boxes
    out = []
    if b is None:
        return out
    for i in range(len(b)):
        c = float(b.conf[i])
        if c < conf_thr:
            continue
        out.append(dict(クラス=result.names[int(b.cls[i])], スコア=round(c, 3)))
    return out

def show(imgs, titles=None, cols=3, height=4.2):
    # 画像を並べて表示する
    imgs = list(imgs)
    if not imgs:
        print("表示する画像がありません。"); return
    cols = min(cols, len(imgs))
    rows = math.ceil(len(imgs) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(height * cols * 1.35, height * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, im, t in zip(axes.ravel(), imgs, titles or [""] * len(imgs)):
        ax.imshow(im); ax.set_title(t, fontsize=10)
    plt.tight_layout(); plt.show()

def predict_closed(path):
    return get_yolo26().predict(path, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]

def predict_open(path, words):
    m = get_yoloe()
    m.set_classes(list(words))
    return m.predict(path, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]

print("OK")

### コードを読んで考えること（1）

上のセルの `dets_of` と `predict_open` を読んで考えよ。

1. `dets_of` は、1枚の画像の検出結果から**何を取り出しているか**。3つ挙げよ。
2. `predict_closed` と `predict_open` の違いは、コードの上では**何行の差**か。
   その行は何をしているか。
3. `CONF` を 0.25 から 0.05 に下げると、検出される数はどう変わるか。**なぜか。**

## 0-5. 画像を置く

画像は **シーンごとにフォルダを分けて** 置く。

```
images/
  01_教室の机上/    IMG_0001.jpg ...
  02_自転車置き場/  ...
  03_自宅の勉強机/  ...
```

Colab は放置すると切れて、アップロードした画像が消える。
**Google ドライブに置くこと**（下の方法B）。

In [ ]:
# ---- 方法A: パソコンから直接アップロードする（1回だけ試すとき用）----
# CONDITION を、いま撮った条件の名前に書き換えてから実行する。
import shutil, os
CONDITION = "01_教室の机上"
(ROOT / CONDITION).mkdir(exist_ok=True, parents=True)

from google.colab import files
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, ROOT / CONDITION / name)
print("保存先:", ROOT / CONDITION)

In [ ]:
# ---- 方法B（推奨）: Google ドライブを使う ----
# 下の3行のコメント（先頭の #）を外してから実行する。

# from google.colab import drive
# drive.mount('/content/drive')
# ROOT = Path('/content/drive/MyDrive/enshu2_images'); ROOT.mkdir(parents=True, exist_ok=True)

def list_scenes(root=None):
    root = Path(root or ROOT)
    return sorted([p for p in root.iterdir() if p.is_dir()])

ALL_PATHS = sum([list_images(s) for s in list_scenes()], []) + list_images(ROOT)
print(f"合計 {len(ALL_PATHS)} 枚")
for s in list_scenes():
    print(f"  {s.name:20s} {len(list_images(s)):3d} 枚")

---
## 0-6. 動かし方は2通りある

Ultralytics のモデルは、**コマンドライン（CLI）**からでも、**Python** からでも動かせる。
**どちらも同じモデル・同じ処理**で、書き方が違うだけである。

|  | CLI | Python |
|---|---|---|
| 書き方 | `yolo predict model=... source=...` | `YOLO(...).predict(...)` |
| 結果の受け取り方 | 画像ファイルとして `runs/` に保存される | **変数として受け取れる** |
| 向いている場面 | 1枚だけ、さっと試したいとき | 何枚も処理して数える・グラフにするとき |
| この演習では | ここで試すだけ | **1節から先はこちらを使う** |

数を数えたりグラフを描いたりするには結果を変数で受け取る必要があるので、
**この演習の本体は Python で書く。** ただし CLI も知っておくと、
研究室のサーバで大量の画像をまとめて処理するときなどに役立つ。

### CLI をノートブックから実行する

Colab や Jupyter のセルの中で CLI を使うときは、**行の先頭に `!` を付ける。**
`{変数名}` と書くと、その中身が展開される。

まずは、クローズドの YOLO26 を CLI で動かしてみる。

In [ ]:
assert ALL_PATHS, "images/ の下に画像を入れてから実行すること。"
img_path = ALL_PATHS[0]        # 試しに使う1枚。何枚目にするかはここで変える
print("使う画像:", img_path)

In [ ]:
# CLI で実行する（先頭の ! を忘れないこと）
!yolo predict model={YOLO26_MODEL} source={img_path} imgsz={IMGSZ} conf={CONF} device={DEVICE} save=True

結果はファイルとして `runs/segment/predict/` の下に保存される。
**画面には出てこない**ので、自分で読み込んで表示する。

In [ ]:
import os

def show_latest_cli_result():
    # CLI が保存した画像のうち、いちばん新しいものを表示する
    files = glob.glob("runs/**/*.jpg", recursive=True) + glob.glob("runs/**/*.png", recursive=True)
    if not files:
        print("runs/ に結果がありません。上の CLI のセルを実行したか確認すること。")
        return
    newest = max(files, key=os.path.getmtime)
    show([to_rgb(cv2.imread(newest))], [f"CLI の結果: {newest}"], cols=1, height=6)

show_latest_cli_result()

### オープン語彙（YOLOE）を CLI で動かす

YOLOE では `classes=` に**探したい物の名前**を書く。

> **紛らわしい点。** ふつうの YOLO で `classes=` と書くと「0番と2番のクラスだけ表示する」
> という**番号での絞り込み**の意味になる。YOLOE では**言葉での指定**になる。
> 同じ名前の引数が、モデルによって違う意味を持つので注意すること。

In [ ]:
!yolo predict model={YOLOE_MODEL} source={img_path} imgsz={IMGSZ} conf={CONF} device={DEVICE} classes="cup" save=True

In [ ]:
show_latest_cli_result()

### 同じことを Python で書く

上の CLI とまったく同じ処理を、Python で書くと次のようになる。
**こちらは結果が変数 `r` に入る**ので、検出された個数を数えたり、
スコアを取り出したりできる。

In [ ]:
# クローズド（YOLO26）
m1 = YOLO(YOLO26_MODEL)
r1 = m1.predict(img_path, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]

# オープン語彙（YOLOE）
m2 = YOLOE(YOLOE_MODEL)
m2.set_classes(["cup"])                       # ← CLI の classes="cup" に相当
r2 = m2.predict(img_path, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]

show([to_rgb(r1.plot()), to_rgb(r2.plot())],
     ["Python: YOLO26", "Python: YOLOE（cup）"], cols=2, height=4.5)

# CLI ではできないこと：結果を数値として取り出す
print("YOLO26 の検出:", dets_of(r1, CONF))
print("YOLOE  の検出:", dets_of(r2, CONF))
print("YOLOE で検出された個数:", len(dets_of(r2, CONF)))

### 書き方の対応表

| CLI | Python | 意味 |
|-----|--------|------|
| `model=yolo26s-seg.pt` | `YOLO("yolo26s-seg.pt")` | どのモデルを使うか |
| `model=yoloe-26s-seg.pt` | `YOLOE("yoloe-26s-seg.pt")` | 同上（オープン語彙） |
| `source=images/a.jpg` | `.predict("images/a.jpg")` | どの画像を入力するか |
| `imgsz=640` | `imgsz=640` | 入力を何画素に揃えるか |
| `conf=0.25` | `conf=0.25` | このスコア未満は捨てる |
| `device=cpu` | `device="cpu"` | CPU と GPU のどちらで動かすか |
| `classes="cup"`（YOLOEのみ） | `m.set_classes(["cup"])` | 探したい物を言葉で指定する |
| `save=True` | （不要。`r.plot()` で画像が得られる） | 結果を画像で保存するか |
| — | `verbose=False` | 実行中のログを出さない |

**0-4 で定義した `predict_closed` と `predict_open` は、上の Python の書き方をまとめただけ**である。
1節から先はこれを使う。中身をもう一度見ておくこと。

### コードを読んで答えること（2）

CLI と Python の対応表を見て、次に答えよ。

1. CLI の `classes="cup"` は、Python では何に相当するか。
2. **同じ `classes=` という名前でも、モデルによって意味が変わる。**
   ふつうの YOLO と YOLOE では、それぞれ何を意味するか。
3. CLI で結果を集計したいとき、なぜ不便なのか。

---
---

# ここから【標準】

**1節から3節までが【標準】である。** 全員がここまでを終わらせること。

---

# 1.【標準】YOLO26 を動かし、何が検出できるか確かめる

> **課題** YOLO26 を試し、枠・クラス名・スコアを確認せよ。
> YOLO26 は MS COCO で学習されている。何が認識対象のクラスなのか調査せよ。
> 各自、様々な画像を収集し、評価をせよ。

## 1-1. まず1枚で動かす

In [ ]:
# img_path は 0-6 で決めたもの。別の画像にしたければここで変える
# img_path = ALL_PATHS[1]

r = predict_closed(img_path)

show([to_rgb(r.plot())], [f"YOLO26: {Path(img_path).name}"], cols=1, height=6)
print("検出:", dets_of(r, CONF))

## 1-2. 認識対象の80クラスを調べる

**この一覧に無い物は、YOLO26 では原理的に検出できない。**

In [ ]:
coco_names = list(get_yolo26().names.values())
print(f"クラス数: {len(coco_names)}\n")
for i in range(0, len(coco_names), 6):
    print("  " + "  ".join(f"{n:<18s}" for n in coco_names[i:i+6]))

## 1-3. 集めた画像すべてに YOLO26 をかける

どんな物が検出され、どんな物が無視されているかを見る。

In [ ]:
imgs, titles = [], []
for p in ALL_PATHS[:9]:                     # 多いときは先頭9枚
    r = predict_closed(p)
    imgs.append(to_rgb(r.plot()))
    names = sorted({d["クラス"] for d in dets_of(r, CONF)})
    titles.append(f"{Path(p).name}\n{names if names else '検出なし'}")
show(imgs, titles, cols=3)

---
# 2.【標準】同じ画像に YOLOE を試し、並べて比較する

> **課題** 同じ画像に YOLOE を試し、結果を並べて比較せよ。

YOLOE は **探したい物を言葉で指定する**。`WORDS` を書き換えて、何度でも試すこと。
指定する言葉は英語のほうが当たりやすいが、日本語も試してみるとよい。

**初回は時間がかかる。** 言葉を扱うための部品（数百MB）をダウンロードするためである。

In [ ]:
WORDS = ["cup", "laptop", "pen"]      # ここを自由に書き換える

imgs, titles = [], []
for p in ALL_PATHS[:3]:
    rc = predict_closed(p)
    ro = predict_open(p, WORDS)
    imgs += [to_rgb(rc.plot()), to_rgb(ro.plot())]
    titles += [f"YOLO26（COCO 80クラス）\n{Path(p).name}",
               f"YOLOE: {', '.join(WORDS)}"]
show(imgs, titles, cols=2, height=4.5)

### 記録すること

- 同じ画像で、検出できた物・できなかった物はどう違ったか。
- YOLOE に指定した言葉のうち、当たったもの・外れたものはどれか。
- **枠の位置は正しいか。** クラス名が合っていても、枠が別の物に乗っていることがある。

---
# 3.【標準】YOLO26 が検出できない物を3つ探す

> **課題** YOLO26 が検出できない物を身の回りから3つ探し、実際に検出されないことを確かめよ。
> 各自、異なる3つの物体を対象とせよ。班で相談し、重複がないようにすること。
> その3つをオープン語彙型で検出せよ。検出できない場合は、その理由の仮説を述べよ。

1-2 で表示した80クラスの一覧に**無い**物を3つ選び、写真を撮ってから実行する。

In [ ]:
# ■ ここは自分たちで決める ■
# 自分が担当する3つの物体を、英語で書く。
# 1-2 の80クラスの一覧に「無い」物を選ぶこと。班の他の人と重ならないようにすること。
# 例：stapler / extension cord / hanger / nail clipper / lunch box / eraser
TARGET_OBJECTS = ["stapler", "extension cord", "hanger"]

assert len(TARGET_OBJECTS) == 3 and all(isinstance(w, str) for w in TARGET_OBJECTS), \
    "3つの物体を、文字列のリストで書くこと"
かぶり = [w for w in TARGET_OBJECTS if w in coco_names]
assert not かぶり, f"{かぶり} は COCO の80クラスに入っている。別の物を選ぶこと。"
print("対象:", TARGET_OBJECTS)

3つの物体を写した画像を `images/対象物/` に入れてから、下を実行する。
左が YOLO26（検出できないはず）、右が YOLOE（言葉で指定）。

In [ ]:
TARGET_DIR = ROOT / "対象物"
target_paths = list_images(TARGET_DIR)

if not target_paths:
    print(f"{TARGET_DIR} に、3つの物体を写した画像を入れること。")
else:
    imgs, titles = [], []
    for p in target_paths[:3]:
        rc = predict_closed(p)
        ro = predict_open(p, TARGET_OBJECTS)
        imgs += [to_rgb(rc.plot()), to_rgb(ro.plot())]
        titles += [f"YOLO26: {sorted({d['クラス'] for d in dets_of(rc, CONF)}) or '検出なし'}",
                   f"YOLOE: {sorted({d['クラス'] for d in dets_of(ro, CONF)}) or '検出なし'}"]
    show(imgs, titles, cols=2, height=4.5)

### 記録すること

- YOLO26 は、その物を**何と誤認したか**。一番近いクラスに引きずられていないか。
- YOLOE は見つけられたか。見つけられないとき、**言葉を変えたら見つかるか。**
- 見つからない理由の仮説を書く。次の4節・5節でその仮説を検証する。

---
---

# ここまでが【標準】

**3節まで終われば、【標準】は完了である。**
次に進む前に、次のものがそろっているか確認すること。

| 節 | そろっているはずのもの |
|----|----------------------|
| 1 | YOLO26 の検出結果の図と、COCO 80クラスの一覧 |
| 2 | 同じ画像を YOLO26 と YOLOE で並べた図 |
| 3 | 担当した3物体の、YOLO26 と YOLOE の結果の図 |

**【標準】だけでも発表とレポートは成立する。** 時間が足りなければ、ここで止めてよい。
そのぶん、検出できた物・できなかった物の観察と考察を厚くすること。

---
---

# ここから【発展】

**【標準】が終わってから取りかかること。**

- **4節**は、語彙を振って検出率を目で数える。【発展】に進むなら全員がここをやる。
- **5節**は、その結果を COCO との「意味の近さ」で説明できるか確かめる。
- **6節**は任意である。**リアルタイムで動くものを作りたい班だけ**取りかかること。

---

# 4.【発展】語彙を振って、得意・不得意を調べる

> **課題** 未学習のクラスについて、得意不得意を調査せよ。
> 様々な語彙を用意して物体検出を試し、うまくいく語彙・いかない語彙に分類せよ。
> 得意不得意の理由について考察せよ。

## 4-1. 調べる語彙を決める

同じ物を指す言葉を変える、抽象度を変える、言語を変える、修飾を足す——など、
**軸を決めてから**語を並べること。行き当たりばったりに並べても比較にならない。

| 軸 | 例 |
|----|----|
| 抽象度 | `container` / `cup` / `mug` / `white ceramic mug` |
| 言語 | `mug` / `マグカップ` |
| 同義語・俗称 | `bicycle` / `bike` / `自転車` |
| 修飾の追加 | `bottle` / `plastic bottle` / `empty plastic bottle` |
| 上位語・下位語 | `furniture` / `chair` / `office chair` |
| COCO からの距離 | `dog`（COCOにある） / `bat`（無い） / `drone`（無い） |

In [ ]:
# ■ ここは自分たちで決める ■
# 調べたい語を6つ以上、リストに並べる。上の表から軸を1つ選んで決めること。
PROMPT_SET = ["container", "cup", "mug", "white ceramic mug", "マグカップ", "tumbler"]

assert len(PROMPT_SET) >= 6 and all(isinstance(w, str) for w in PROMPT_SET), \
    "語を文字列で6つ以上書くこと"
print(len(PROMPT_SET), "個の語を調べる")

## 4-2. 語ごとに結果を並べて、目で確かめる

**1語ずつ指定すること。** 複数の語を同時に渡すと、重なった枠のどちらかが消えてしまい、
1語ずつ試したときと違う結果になる。

下のセルを `WORD` を変えながら繰り返し実行し、**その語で何枚正しく検出できたかを目で数える。**
自動で数えた枚数も出るが、それは「何かが検出された枚数」であって、
**枠が正しい物に乗っているかまでは見ていない。** そこは自分の目で確かめること。

In [ ]:
WORD  = PROMPT_SET[0]              # 調べたい語をここで選ぶ
PATHS = ALL_PATHS[:10]             # 見る画像（多すぎると数えられない）

imgs, titles, auto = [], [], 0
for p in PATHS:
    r = predict_open(p, [WORD])
    d = dets_of(r, CONF)
    if d:
        auto += 1
    imgs.append(to_rgb(r.plot()))
    titles.append(f"{Path(p).name}  {len(d)}件")
show(imgs, titles, cols=3, height=3.6)

print(f"語『{WORD}』：{len(PATHS)} 枚中、何かが検出された画像は {auto} 枚（自動集計）")
print("→ 上の図を見て、**枠が正しい物に乗っている枚数**を自分で数えること。")

### コードを読んで答えること（3）

上のセルは `predict_open(p, [WORD])` と、**語を1つだけ**渡している。
リストなので `["cup", "mug"]` のように複数渡すこともできる。

1. それでも1語ずつ渡しているのはなぜか。4-2 の本文にある `agnostic_nms` の説明を読んで答えよ。
2. まとめて渡したとき、**消えるのは何か**。スコアは変わるか。
3. `auto` が数えているのは何枚か。それは「正しく検出できた枚数」と同じか。**違うなら、なぜ違うか。**

## 4-3. 数えた結果を書き込む

すべての語について 4-2 を実行し、目で数えた枚数を下に書き込む。

In [ ]:
TOTAL = len(PATHS)          # 4-2 で見た枚数（自動で入る）
print("見た枚数:", TOTAL)

# 語ごとに「正しく検出できた枚数」を目で数えて書く
COUNTED = {
    "container":         3,
    "cup":               8,
    "mug":               9,
    "white ceramic mug": 6,
    "マグカップ":          2,
    "tumbler":           4,
}

for w in PROMPT_SET:
    assert w in COUNTED, f"『{w}』の枚数がまだ書かれていない"
print("記入済み:", len(COUNTED), "語")

In [ ]:
# 検出率を求める。正しく検出できた枚数 ok を、試した枚数 total で割る。
# total が 0 のときに 0 で割らないよう、if で分けてある。

def detection_rate(ok, total):
    if total == 0:
        return float("nan")
    else:
        return ok / total


# --- 確認：0.8 と表示される ---
print(detection_rate(8, 10))

## 4-4. 語ごとの検出率を並べる

In [ ]:
rows = []
for w in PROMPT_SET:
    rows.append({"語": w, "正しく検出できた枚数": COUNTED[w], "試した枚数": TOTAL,
                 "検出率": round(detection_rate(COUNTED[w], TOTAL), 3)})
rate_df = pd.DataFrame(rows).sort_values("検出率", ascending=False).reset_index(drop=True)
display(rate_df)

plt.figure(figsize=(1.5 * len(rate_df) + 3, 4.5))
plt.bar(rate_df["語"], rate_df["検出率"], color="#27509B")
plt.ylim(0, 1.05); plt.ylabel("検出率"); plt.xticks(rotation=20, ha="right")
plt.title(f"語による検出率の違い（{TOTAL}枚中）")
plt.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()

print("うまくいった語:", list(rate_df[rate_df["検出率"] >= 0.7]["語"]))
print("うまくいかない語:", list(rate_df[rate_df["検出率"] < 0.4]["語"]))

### 数字の読み方に注意

10枚しか見ていないので、**1枚の判定が変わるだけで検出率は 0.1 動く。**
`0.7` と `0.8` の差を「差がある」と書いてはいけない。
**うまくいく／いかないの大きな分かれ目** だけを論じること。

---
# 5.【発展】MS COCO のクラスとの「意味の近さ」を測る

> **ヒント（課題文より）** MS COCO のクラスとの関係。
> 人間の感覚で近い／遠いと思った単語。画像言語基盤モデル（CLIP）が近い／遠いと思った単語。

4節で「うまくいく語」と「いかない語」が分かった。ではその差は何から来るのか。
ひとつの仮説は「 **MS COCO のクラスに意味が近い語ほど、うまく検出できる** 」である。

これは確かめられる。YOLOE は言葉を **数のならび（ベクトル）** に変換してから使っているので、
**そのベクトルを取り出して、COCO の80クラスとの近さを測ればよい。**

## 5-1. 言葉をベクトルに変換する

`get_text_pe` を使うと、YOLOE が実際に使っているベクトルがそのまま取り出せる。

In [ ]:
def text_vectors(words):
    # 言葉のリストを、YOLOE が使っているベクトルに変換する
    m = get_yoloe()
    pe = m.get_text_pe(list(words))
    v = pe.detach().cpu().numpy()
    return v.reshape(len(words), -1)

v = text_vectors(["dog", "cat", "bicycle"])
print("ベクトルの形:", v.shape, " ← (語の数, 1語あたりの次元数)")

## 5-2. 2つのベクトルの近さを測る

**コサイン類似度**を使う。2つのベクトルの向きがどれだけ揃っているかを表す値で、
向きが同じなら 1、直角なら 0 になる。

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

- 分子 $\mathbf{a}\cdot\mathbf{b}$ は内積。`np.dot(a, b)` で計算できる。
- 分母の $\|\mathbf{a}\| はベクトルの長さ。`np.linalg.norm(a)` で計算できる。

In [ ]:
# コサイン類似度を求める。内積を、2つのベクトルの長さの積で割っている。
# 長さで割ることで、ベクトルの「大きさ」を打ち消し、「向き」だけを比べられる。

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# --- 確認：1.0  0.0  と表示される ---
p = np.array([1.0, 0.0])
q = np.array([2.0, 0.0])      # p と同じ向き
r = np.array([0.0, 3.0])      # p と直角
print(round(cosine_similarity(p, q), 3), round(cosine_similarity(p, r), 3))

## 5-3. COCO の80クラスの中から、いちばん近いものを探す

In [ ]:
# ベクトル vec に、いちばん近い COCO クラスを探す。
# coco_vecs[i] が coco_names[i] のベクトルになっている。
# for で全部の類似度を計算し、いちばん大きかったものを覚えている。

def nearest_coco(vec, coco_vecs, coco_names):
    best_sim = -1.0
    best_name = ""
    for i in range(len(coco_names)):
        sim = cosine_similarity(vec, coco_vecs[i])
        if sim > best_sim:
            best_sim = sim
            best_name = coco_names[i]
    return best_name, best_sim


# --- 確認：cat 1.0 と表示される ---
テスト用ベクトル = np.array([[1.0, 0.0], [0.0, 1.0]])
名前, 近さ = nearest_coco(np.array([0.0, 2.0]), テスト用ベクトル, ["dog", "cat"])
print(名前, round(float(近さ), 3))

### コードを読んで答えること（4）

`text_vectors`・`cosine_similarity`・`nearest_coco` を読んで、次に答えよ。

1. `cosine_similarity` は、なぜ内積を**2つの長さの積で割っている**のか。
   割らなかったら、何を比べていることになるか。
2. `nearest_coco` は、1つの語について**何回**コサイン類似度を計算しているか。
3. `text_vectors` が返すベクトルは、**どこから来ているか**。
   （`get_text_pe` は誰のメソッドか、を見ること）

## 5-4. 「COCO との近さ」と「検出率」を並べる

仮説が正しければ、**COCO に近い語ほど検出率が高い**はずである。

In [ ]:
coco_vecs = text_vectors(coco_names)         # 80クラスのベクトル（少し時間がかかる）
word_vecs = text_vectors(PROMPT_SET)         # 調べた語のベクトル

rows = []
for w, v in zip(PROMPT_SET, word_vecs):
    name, sim = nearest_coco(v, coco_vecs, coco_names)
    rows.append({"語": w, "いちばん近いCOCOクラス": name,
                 "COCOとの近さ": round(float(sim), 3),
                 "検出率": round(detection_rate(COUNTED[w], TOTAL), 3)})
sim_df = pd.DataFrame(rows).sort_values("COCOとの近さ", ascending=False).reset_index(drop=True)
display(sim_df)

plt.figure(figsize=(7, 5.5))
plt.scatter(sim_df["COCOとの近さ"], sim_df["検出率"], s=90, color="#27509B")
for _, r in sim_df.iterrows():
    plt.annotate(r["語"], (r["COCOとの近さ"], r["検出率"]),
                 textcoords="offset points", xytext=(6, 6), fontsize=10)
plt.xlabel("COCO のクラスとの近さ（コサイン類似度の最大値）")
plt.ylabel("検出率")
plt.ylim(-0.05, 1.1); plt.grid(alpha=.3)
plt.title("COCO に近い語ほど検出できるか")
plt.tight_layout(); plt.show()

## 5-5. 人間の感覚とも比べる

**自分たちが「COCO のこのクラスに近い」と思った語**と、
**モデルが近いと判断した語**は一致しただろうか。

下のセルで、任意の2語の近さを直接測れる。
自分の感覚と食い違う組を3つ以上見つけて、レポートに載せること。

In [ ]:
PAIRS = [
    ("dog", "cat"),
    ("dog", "bicycle"),
    ("mug", "cup"),
    ("マグカップ", "cup"),
    ("drone", "airplane"),
    ("stapler", "scissors"),
]

words = sorted({w for pair in PAIRS for w in pair})
vecs = dict(zip(words, text_vectors(words)))
for a, b in PAIRS:
    print(f"  {a:20s} - {b:20s}  {cosine_similarity(vecs[a], vecs[b]):.3f}")

### 考察してほしいこと

- **COCO に近い語ほど検出率が高い、という仮説は成り立ったか。** 成り立たない語はどれか。
  その語は何が特別だったのか。
- 同じ物を指す日本語と英語で、ベクトルの近さと検出率はどう違ったか。
  なぜそうなるのか（学習に使われた説明文は何語で書かれていたか）。
- 抽象度を上げた語（`container` など）は、検出率が高く出ることがある。
  それは**本当に正しく検出できている**のか、それとも**何にでも当たっているだけ**なのか。
  4-2 の図に戻って確かめること。
- 「言葉で指定できる」ことの限界はどこにあるか。
  どんな物なら言葉で指定でき、どんな物なら難しいか。

---
---

# 6.【発展・任意】リアルタイムで動くものを作る

ここまでは、**撮りためた画像を後からまとめて処理**してきた。
YOLOE は速いので、**カメラの映像をその場で処理する**こともできる。
「言葉で指定した物が映ったら知らせる」といったものが作れる。

## 6-1. なぜ Colab ではできないのか

Colab は **Google のサーバの上で動いている**。手元のパソコンのカメラは繋がっていない。
リアルタイムに動かすなら、**モデルを自分のパソコンで動かす必要がある。**

**付録 の手順で、自分のパソコンに環境を作ること。** そのうえで、下のコードを動かす。

> GPU が無くても動く。ただし `s` のモデルで数フレーム/秒くらいなので、
> 遅ければ `n` に落とすか、`imgsz` を小さくすること。
> **ここだけは `n` を使ってよい**（速さを見るのが目的なので）。

## 6-2. いちばん短い書き方

CLI なら1行で動く。`source=0` が内蔵カメラである。

```
yolo predict model=yoloe-26s-seg.pt source=0 classes="cup" show=True
```

Python なら次のとおり。**下のセルはスクリプトを書き出すだけ**で、Colab では実行しない。
ダウンロードして、自分のパソコンで `python realtime_demo.py` と打つこと。

In [ ]:
%%writefile realtime_demo.py
# YOLOE をカメラ映像にリアルタイムで適用する例
# 使い方: python realtime_demo.py
#   q キーで終了
import cv2
from ultralytics import YOLOE

MODEL = "yoloe-26s-seg.pt"    # 遅ければ "yoloe-26n-seg.pt" に落とす
WORDS = ["cup", "bottle", "smartphone"]   # 探したい物を言葉で書く
CAM   = 0                     # 内蔵カメラ。外付けなら 1, 2, ...
IMGSZ = 480                   # 小さいほど速い
CONF  = 0.25


def main():
    model = YOLOE(MODEL)
    model.set_classes(WORDS)          # 言葉で「何を探すか」を指定
    print("探すもの:", WORDS)
    print("q キーで終了")

    cap = cv2.VideoCapture(CAM)
    if not cap.isOpened():
        raise SystemExit("カメラを開けません。CAM の番号を変えてみること。")

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        # 1フレームを推論する。verbose=False にしないとログが流れ続ける
        r = model.predict(frame, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
        cv2.imshow("YOLOE realtime  (q で終了)", r.plot())
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

## 6-3. 何を作るか

上のコードは「映して枠を出す」だけである。ここから先は自由に作ってよい。
**大がかりなものでなくてよい。** 発想が評価される。

| 方向 | 例 |
|------|-----|
| 見つけたら知らせる | 指定した物が映ったら音を鳴らす／画面に大きく出す |
| 数える | 映っている数を数え続け、増減を記録する |
| 言葉を切り替える | キーを押すと探す物が切り替わる（`set_classes` を呼び直す） |
| 時間を測る | その物が画面にあった時間を計る |
| 速さを調べる | `imgsz` やモデルの大きさを変えて、フレームレートがどう変わるか測る |

> **`set_classes` は呼ぶたびに時間がかかる。** 毎フレーム呼ばないこと。
> 探す物を切り替えるときだけ呼ぶようにする。**この違いを実測すると、それ自体が結果になる。**

## 6-4. レポートに書くこと

- 何を作ったか。**画面の写真か、短い動画**を載せること。
- **どのくらいの速さで動いたか。** 1秒あたり何フレームか、モデルの大きさや `imgsz` を変えるとどう変わったか。
- 画像をまとめて処理したとき（1〜5節）と比べて、**リアルタイムでは何が難しかったか。**
- うまく動かなかったなら、**どこで詰まったか**を書く。それも結果である。


---
# 付録：自分のパソコンで動かす（余裕がある人向け）

**まず Colab で最後まで通してから試すこと。** 環境構築でつまずくと時間がなくなる。

Anaconda Prompt（Windows）またはターミナル（macOS）で、次を**順に**実行する。

```
conda create -n enshu python=3.11 git -y
conda activate enshu
pip install "ultralytics>=8.4.0" pandas matplotlib japanize-matplotlib notebook ipykernel
pip install git+https://github.com/ultralytics/CLIP.git
python -m ipykernel install --user --name enshu2 --display-name "enshu"
```

**`git` を conda に含めているのが要点。** これが無いと、
言葉で指定する機能を初めて使うときに `Cannot find command 'git'` で止まる。

**このノートブックを置いたフォルダに移動してから** Jupyter を起動する。

```
cd （このノートブックのあるフォルダ）
jupyter notebook
```

- **場所が大事な理由**：YOLOE はテキストエンコーダ（数百MB）を
  **その時のカレントディレクトリ**にダウンロードする。毎回違う場所で起動すると、そのたびに落とし直す。
- **カーネルを確認すること**：右上のカーネル名が `enshu` になっているか見る。
  `Python 3` のままだと `ModuleNotFoundError: No module named 'ultralytics'` になる。
  Kernel → Change kernel → enshu で切り替える。
- ローカルで動かす場合、`from google.colab import ...` のセルは実行しないこと。
  `ROOT = Path("images")` のまま、そのフォルダに画像を置けばよい。

---
## つまずいたときは

| 症状 | 対処 |
|------|------|
| `yoloe-26s-seg.pt` が読めない | `ultralytics` が 8.4.0 未満。更新してランタイム再起動 |
| `Cannot find command 'git'` | 0-1 の CLIP のセルを実行していない。上に戻る |
| `set_classes` で止まる | 初回はテキストエンコーダ（数百MB）を落としている。数分待つ |
| CPU で遅すぎる | `IMGSZ` を 512 や 416 に下げる。モデルは `s` のまま |
| 画像が見つからない | HEIC 形式かもしれない。iPhone の設定を「互換性優先」に |
| セッションが切れて画像が消えた | 画像は Google ドライブに置くこと（0-5 の方法B） |
| 図の日本語が □ | `!pip install japanize-matplotlib` してランタイムを再起動 |
